In [1]:
import gseapy as gp
import pandas as pd
import numpy as np

import os
import openpyxl
from sklearn.model_selection import train_test_split

import gc

/home/vasileioubill95/miniconda3/lib/python3.12/site-packages/numpy/_core/getlimits.py:548: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


In [2]:
os.chdir("/mnt/c/Users/vasileioubill95/Desktop/Projects/lncAPNet_Prostate")

In [3]:
### PATHWAY ENRICHMENT

In [4]:
# Load the Excel files
prostate = pd.read_excel("Prostate_NetBID2/Driver_output/Prostate_NetBID2/DATA/ms_tab_ens.xlsx")

# Filtering based on conditions
pos = prostate[(prostate["adj.P.Val.U.Vs.M_DA"] < 0.05) & (prostate["logFC.U.Vs.M_DA"] > 0) & (prostate["Size"] > 30)]
neg = prostate[(prostate["adj.P.Val.U.Vs.M_DA"] < 0.05) & (prostate["logFC.U.Vs.M_DA"] < 0) & (prostate["Size"] > 30)]

# Row-bind (combine) the results
ms_tab = pd.concat([pos, neg], axis=0, ignore_index=True)

# Get unique values from the 'originalID' column
ms_tab = list(ms_tab['hgnc_symbol.y'].unique())
ms_tab.sort()

# Display the result
len(ms_tab)

4331

In [7]:
# For Gene Ontology
# Run ORA with a custom GMT file
enr = gp.enrichr(
    gene_list=ms_tab, 
    gene_sets="data/EnrichR/both/merged_GO_Biological_Process_2021.gmt",  # Path to your custom GMT file
    organism="Human", 
    outdir="data/EnrichR/both/merged_GO_Biological_Process_2021_res",
    cutoff=0.1
)

results_go = enr.results

results_go_filt = results_go[(results_go['P-value'] < 0.05)]
len(results_go_filt)

926

In [10]:
results_go_filt = results_go[(results_go['Adjusted P-value'] < 0.05)]
len(results_go_filt)

140

In [11]:
results_go_filt['Genes'] = results_go_filt['Genes'].str.split(';')

results_go_filt_n = results_go_filt.explode("Genes").pivot_table(index="Term", columns="Genes", aggfunc="size", fill_value=0).reset_index()
results_go_filt_n = results_go_filt_n.set_index('Term')

results_go_filt_n.to_excel("PASNet/Input/GO/pt_fixed_ens.xlsx")

/tmp/ipykernel_99/3164848716.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_go_filt['Genes'] = results_go_filt['Genes'].str.split(';')


In [12]:
# For KEGG
# Run ORA with a custom GMT file
enr = gp.enrichr(
    gene_list=ms_tab, 
    gene_sets="data/EnrichR/both/merged_KEGG_2021_Human.gmt",  # Path to your custom GMT file
    organism="Human", 
    outdir="data/EnrichR/both/merged_KEGG_2021_Human_res",
    cutoff=0.1
)

results_kegg = enr.results

results_kegg_filt = results_kegg[(results_kegg['P-value'] < 0.05)]
len(results_kegg_filt)

56

In [13]:
results_kegg_filt = results_kegg[(results_kegg['Adjusted P-value'] < 0.05)]
len(results_kegg_filt)

14

In [14]:
# For REACTOME
# Run ORA with a custom GMT file
enr = gp.enrichr(
    gene_list=ms_tab, 
    gene_sets="data/EnrichR/both/merged_Reactome_2022.gmt",  # Path to your custom GMT file
    organism="Human", 
    outdir="data/EnrichR/both/merged_Reactome_2022_res",
    cutoff=0.1
)

results_reactome = enr.results

results_reactome_filt = results_reactome[(results_reactome['P-value'] < 0.05)]
len(results_reactome_filt)

314

In [15]:
results_reactome_filt = results_reactome[(results_reactome['Adjusted P-value'] < 0.05)]
len(results_reactome_filt)

92

In [16]:
# For Wikipathways
# Run ORA with a custom GMT file
enr = gp.enrichr(
    gene_list=ms_tab, 
    gene_sets="data/EnrichR/both/merged_WikiPathway_2021_Human.gmt",  # Path to your custom GMT file
    organism="Human", 
    outdir="data/EnrichR/both/merged_WikiPathway_2021_res",
    cutoff=0.1
)
results_wiki = enr.results

results_wiki_filt = results_wiki[(results_wiki['P-value'] < 0.05)]
len(results_wiki_filt)

93

In [17]:
results_wiki_filt = results_wiki[(results_wiki['Adjusted P-value'] < 0.05)]
len(results_wiki_filt)

0

In [18]:
df_combined = pd.concat([results_kegg_filt, results_reactome_filt, results_wiki_filt], axis=0, ignore_index=True)

In [19]:
df_combined['Genes'] = df_combined['Genes'].str.split(';')

df_combined_n = df_combined.explode("Genes").pivot_table(index="Term", columns="Genes", aggfunc="size", fill_value=0).reset_index()
df_combined_n = df_combined_n.set_index('Term')

df_combined_n.to_excel("PASNet/Input/REST/pt_fixed_ens.xlsx")

In [20]:
# Open a text file in write mode
with open("../../output.txt", "w") as file:
    # Iterate through the list and write each item to the file
    for item in df_combined_n.columns:
        file.write(item + "\n")

print("List has been written to 'output.txt'.")

List has been written to 'output.txt'.


In [21]:
# Garbage collection
gc.collect()

working_dir = "/mnt/c/Users/vasileioubill95/Desktop/Projects/lncAPNet_Prostate/"

# Load activity data
activity = pd.read_csv(working_dir + "data/activity.csv")
activity['Unnamed: 0'] = activity['Unnamed: 0'].str.replace('_TF', '').str.replace('_SIG', '').str.replace('_LNC', '')
activity = activity.drop_duplicates(subset=['Unnamed: 0']).set_index('Unnamed: 0').T

# Load metadata
metadata = pd.read_csv(working_dir + "data/metadata.csv", index_col=0)
metadata['Status'] = metadata['Status'].replace({'Progressive': '1', 'Mild': '0'})

# Load pathway data
pt_rest = pd.read_excel(working_dir + "PASNet/Input/REST/pt_fixed_ens.xlsx", index_col=0)
pt_go = pd.read_excel(working_dir + "PASNet/Input/GO/pt_fixed_ens.xlsx", index_col=0)

# Subset activity data
activity_go = activity.loc[:, activity.columns.isin(pt_go.columns)]
activity_rest = activity.loc[:, activity.columns.isin(pt_rest.columns)]

# Merge with metadata
activity_go = activity_go.merge(metadata, left_index=True, right_index=True)
activity_rest = activity_rest.merge(metadata, left_index=True, right_index=True)

# Split into train and test sets
pretrain_go, validation_go = train_test_split(activity_go, test_size=0.2, random_state=123, stratify=activity_go['Status'])
pretrain_rest, validation_rest = train_test_split(activity_rest, test_size=0.2, random_state=123, stratify=activity_rest['Status'])

train_go, validation_grinding_go = train_test_split(pretrain_go, test_size=0.2, random_state=123, stratify=pretrain_go['Status'])
train_rest, validation_grinding_rest = train_test_split(pretrain_rest, test_size=0.2, random_state=123, stratify=pretrain_rest['Status'])

# Save to Excel
train_go.to_excel(working_dir + "PASNet/Input/GO/Training_ens.xlsx", index=True)
validation_go.to_excel(working_dir + "PASNet/Input/GO/Validation_ens.xlsx", index=True)
validation_grinding_go.to_excel(working_dir + "PASNet/Input/GO/Validation_grinding_ens.xlsx", index=True)
train_rest.to_excel(working_dir + "PASNet/Input/REST/Training_ens.xlsx", index=True)
validation_rest.to_excel(working_dir + "PASNet/Input/REST/Validation_ens.xlsx", index=True)
validation_grinding_rest.to_excel(working_dir + "PASNet/Input/REST/Validation_grinding_ens.xlsx", index=True)

In [22]:
pt_go_filt = pt_go.loc[:, pt_go.columns.isin(activity_go.columns)]
pt_rest_filt = pt_rest.loc[:, pt_rest.columns.isin(activity_rest.columns)]

In [23]:
pt_rest_filt.to_excel("PASNet/Input/REST/pt_fixed_ens.xlsx")
pt_go_filt.to_excel("PASNet/Input/GO/pt_fixed_ens.xlsx")